# Imports

In [ ]:
import lyricsgenius

import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import f1_score

c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Functions

Will compare performance based on different section splitting methods

In [ ]:
def split_into_sections(lyrics:str):
    """
    This version of splitting splits based off the line not the verse
    """
    sections = re.split(r"\n\s*\n|\[.*?\]", lyrics)
    return [s.strip() for s in sections if len(s.strip()) > 0]

# def split_into_sections(lyrics: str) -> List[str]:
#     """
#     Splits lyrics into sections (verses / choruses)
#     using blank lines or [SECTION] markers.
#     """
#     sections = re.split(r"\n\s*\n|\[.*?\]", lyrics)

#     return [
#         section.strip()
#         for section in sections
#         if len(section.strip()) > 0
#     ]

def get_tokenizer(model_name='roberta-base'):
    """
    Loads the tokenizer corresponding to the encoder model
    """
    return AutoTokenizer.from_pretrained(model_name,use_fast=True)
    
#Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}   

# def chunk_lyrics(lyrics, max_tokens=256):
#     chunks = split_into_verses(lyrics)

In [ ]:
#Classes

#Dataset
class LyricsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item
    

#Model
class LyricRoBERTa(nn.Module):
    def __init__(self, model_name="roberta-base", num_labels=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            self.encoder.config.hidden_size,
            num_labels
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(cls_embedding))

        return logits


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = LyricRoBERTa()

train_ds = LyricsDataset(train_texts, train_labels, tokenizer)
val_ds = LyricsDataset(val_texts, val_labels, tokenizer)

args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

In [10]:
genius = lyricsgenius.Genius("rPVs-pFT7GhBfTxhpu-ISnNCcvCsbRt8wIwhkkEXovrADgSBQfkndQW7Ge22R5Ts", timeout=60)
song = genius.search_song('Birds Dont sing', 'Clipse')
lyrics = song.lyrics

Searching for "Birds Dont sing" by Clipse...
Done.


In [11]:
print(lyrics)

[Intro]
(
Birds don't, birds don't, birds don't, birds don't
)

[Verse 1: Pusha T]
Lost in emotion, mama's youngest
Tryna navigate life without my compass
Some experience death and feel numbness
But not me, I felt it all and couldn't function
Seein' you that day
Tellin' you my plans but I was leavin' you that day
It was in God's hands, Ye was at Elon's waiting to get with me
On my way to Texas, that's when Virginia hit me
And I realized in that instant
Our last conversation, you was against it
Told you I was going to Turks for Thanksgiving
I heard what I wanted to hear but didn't listen
You said you told Gene that Bup needed forgiveness
I see you went to DD's and stuffed both her fridges
You even told Dad you wished y'all never splitted
See, you were checkin' boxes, I was checkin' my mentions
Sayin' you was tired but not ready to go
Basically was dying without letting me know
I loved you met Nige, hate that he won't remember you
Two things that break my heart is what Novembers do
And T

In [13]:
new_lyrics = split_into_sections(lyrics)
print(new_lyrics)

["(\nBirds don't, birds don't, birds don't, birds don't\n)", "Lost in emotion, mama's youngest\nTryna navigate life without my compass\nSome experience death and feel numbness\nBut not me, I felt it all and couldn't function\nSeein'\u205fyou\u205fthat\u205fday\nTellin' you my\u205fplans but I\u205fwas leavin' you that day\nIt was in God's hands, Ye was at Elon's waiting to get with me\nOn my way to Texas, that's when Virginia hit me\nAnd I realized in that instant\nOur last conversation, you was against it\nTold you I was going to Turks for Thanksgiving\nI heard what I wanted to hear but didn't listen\nYou said you told Gene that Bup needed forgiveness\nI see you went to DD's and stuffed both her fridges\nYou even told Dad you wished y'all never splitted\nSee, you were checkin' boxes, I was checkin' my mentions\nSayin' you was tired but not ready to go\nBasically was dying without letting me know\nI loved you met Nige, hate that he won't remember you\nTwo things that break my heart is 